# Project 58 — Local Calendar Planner Agent
## Conflict Detection → Smart Scheduling → Daily Briefs

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Calendar Data Model

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

calendar = [
    {"id": 1, "title": "Team Standup",       "day": "Mon-Fri", "start": "09:00", "end": "09:30", "priority": "high"},
    {"id": 2, "title": "Sprint Planning",    "day": "Monday",  "start": "10:00", "end": "12:00", "priority": "high"},
    {"id": 3, "title": "Product Review",     "day": "Tuesday", "start": "10:00", "end": "11:00", "priority": "medium"},
    {"id": 4, "title": "1:1 with Manager",   "day": "Wednesday","start": "14:00", "end": "14:30", "priority": "high"},
    {"id": 5, "title": "Focus Time",         "day": "Thursday", "start": "13:00", "end": "16:00", "priority": "medium"},
    {"id": 6, "title": "Team Lunch",         "day": "Friday",   "start": "12:00", "end": "13:00", "priority": "low"},
    {"id": 7, "title": "Code Review Session","day": "Tue,Thu",  "start": "15:00", "end": "16:00", "priority": "medium"},
]

print(f"Calendar has {len(calendar)} recurring events:")
for e in calendar:
    print(f"  {e['day']:<12} {e['start']}-{e['end']}  [{e['priority']}]  {e['title']}")

## Step 2 — Conflict Detector

In [ ]:
class ConflictReport(BaseModel):
    conflicts: list[str]
    overloaded_days: list[str]
    free_blocks: list[str] = Field(description="Available 1+ hour blocks per day")
    utilization: float = Field(description="% of 9-5 schedule occupied")

detector = llm.with_structured_output(ConflictReport)

report = detector.invoke(
    f"Analyze this weekly calendar for conflicts, overloaded days, and free blocks.\n"
    f"Working hours: 9:00-17:00.\n\nCalendar:\n{json.dumps(calendar, indent=2)}"
)

print("CONFLICT REPORT")
print("=" * 50)
print(f"Utilization: {report.utilization:.0%}")
if report.conflicts:
    for c in report.conflicts:
        print(f"  ⚠ {c}")
else:
    print("  No conflicts found")
print(f"\nOverloaded days: {report.overloaded_days}")
print(f"\nFree blocks:")
for fb in report.free_blocks:
    print(f"  ✓ {fb}")

## Step 3 — Smart Meeting Scheduler

In [ ]:
class MeetingSlot(BaseModel):
    day: str
    start: str
    end: str
    score: float = Field(ge=0, le=1, description="Suitability 0-1")
    reasoning: str

class ScheduleProposal(BaseModel):
    request: str
    proposed_slots: list[MeetingSlot]
    best_slot: str
    conflicts_avoided: list[str]

scheduler = llm.with_structured_output(ScheduleProposal)

requests = [
    "Schedule a 90-minute design review with 3 people this week",
    "Find a 2-hour deep work block on Wednesday or Thursday",
    "Add a 30-minute daily check-in, preferably after standup",
    "Schedule a 1-hour all-hands meeting avoiding focus time",
]

for req in requests:
    proposal = scheduler.invoke(
        f"Calendar:\n{json.dumps(calendar, indent=2)}\n\n"
        f"Working hours: 9:00-17:00\nRequest: {req}"
    )
    print(f"\nRequest: {req}")
    print(f"Best slot: {proposal.best_slot}")
    for slot in proposal.proposed_slots[:3]:
        print(f"  • {slot.day} {slot.start}-{slot.end} (score={slot.score:.0%}) — {slot.reasoning}")

## Step 4 — Daily Brief Generator

In [ ]:
brief_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a friendly daily brief for the given day's schedule. "
     "Include: greeting, today's events with times, preparation reminders, "
     "and suggested focus areas for free blocks."),
    ("human", "Day: {day}\nCalendar:\n{events}")
])
brief_chain = brief_prompt | llm | StrOutputParser()

for day in ["Monday", "Wednesday", "Friday"]:
    day_events = [e for e in calendar if day in e["day"] or "Mon-Fri" in e["day"]]
    brief = brief_chain.invoke({
        "day": day,
        "events": json.dumps(day_events, indent=2),
    })
    print(f"\n{'='*50}")
    print(f"DAILY BRIEF — {day}")
    print("=" * 50)
    print(brief[:400])

## What You Learned
- **Calendar conflict detection** with structured analysis
- **Smart scheduling** scored by suitability
- **Daily brief generation** from calendar data
- **Time-slot optimization** with LLM reasoning